# ENV ET OUTILS

In [19]:
import torch # pour la manipulation des tenseurs
import torch.nn as nn # pour la création de modèles de réseaux de neurones
import torch.optim as optim # pour l'optimisation des paramètres du modèle

# 1. Nos données d'entraînement (Texte, Tags)
# On utilise un exemple simple en français pour commencer
training_data = [
    ("Amadou étudie le bambara".split(), ["NOM", "VERBE", "DET", "NOM"]),
    ("le modèle comprend les phrases".split(), ["DET", "NOM", "VERBE", "DET", "NOM"])
]

# 2. Création du dictionnaire de mots (Word to Index)
word_to_ix = {}
for sentence, tags in training_data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

# On ajoute un token spécial pour les mots inconnus (Out Of Vocabulary)
word_to_ix["<UNK>"] = len(word_to_ix)

# 3. Création du dictionnaire d'étiquettes (Tag to Index)
tag_to_ix = {"NOM": 0, "VERBE": 1, "DET": 2}

print("Dictionnaire des mots (word_to_ix) :")
print(word_to_ix)
print("\nDictionnaire des étiquettes (tag_to_ix) :")
print(tag_to_ix)

"""
en gros le dictionnaire des mots (word_to_ix) est utilisé pour convertir les mots en indices numériques, ce qui est nécessaire pour l'entrée dans le modèle de réseau de neurones. 
Chaque mot unique dans les données d'entraînement se voit attribuer un index unique. 
Le token spécial "<UNK>" est utilisé pour représenter les mots qui ne sont pas présents dans le dictionnaire, ce qui permet au modèle de gérer les mots inconnus lors de l'inférence.    
"""

Dictionnaire des mots (word_to_ix) :
{'Amadou': 0, 'étudie': 1, 'le': 2, 'bambara': 3, 'modèle': 4, 'comprend': 5, 'les': 6, 'phrases': 7, '<UNK>': 8}

Dictionnaire des étiquettes (tag_to_ix) :
{'NOM': 0, 'VERBE': 1, 'DET': 2}


'\nen gros le dictionnaire des mots (word_to_ix) est utilisé pour convertir les mots en indices numériques, ce qui est nécessaire pour l\'entrée dans le modèle de réseau de neurones. \nChaque mot unique dans les données d\'entraînement se voit attribuer un index unique. \nLe token spécial "<UNK>" est utilisé pour représenter les mots qui ne sont pas présents dans le dictionnaire, ce qui permet au modèle de gérer les mots inconnus lors de l\'inférence.    \n'

In [20]:
# Fonction pour convertir une liste de mots en tenseur d'index
def prepare_sequence(seq, to_ix):
    idxs = []
    for w in seq:
        if w in to_ix:
            idxs.append(to_ix[w])
        else:
            idxs.append(to_ix["<UNK>"])  # Si le mot est inconnu
    return torch.tensor(idxs, dtype=torch.long)

# Test de la fonction
exemple_phrase = "Amadou comprend le bambara".split()
tensor_phrase = prepare_sequence(exemple_phrase, word_to_ix)

print("Phrase d'origine :", exemple_phrase)
print("Tenseur PyTorch généré :", tensor_phrase)

"""
ici , la fonction prepare_sequence prend une séquence de mots et le dictionnaire word_to_ix comme entrées.
Elle convertit chaque mot en son index correspondant dans le dictionnaire.
Si un mot n'est pas trouvé dans le dictionnaire, il est remplacé par l'index du token spécial "<UNK>".
Le résultat est un tenseur PyTorch contenant les indices des mots, prêt à être utilisé comme entrée pour un modèle de réseau de neurones.

"""

Phrase d'origine : ['Amadou', 'comprend', 'le', 'bambara']
Tenseur PyTorch généré : tensor([0, 5, 2, 3])


'\nici , la fonction prepare_sequence prend une séquence de mots et le dictionnaire word_to_ix comme entrées.\nElle convertit chaque mot en son index correspondant dans le dictionnaire.\nSi un mot n\'est pas trouvé dans le dictionnaire, il est remplacé par l\'index du token spécial "<UNK>".\nLe résultat est un tenseur PyTorch contenant les indices des mots, prêt à être utilisé comme entrée pour un modèle de réseau de neurones.\n\n'

In [21]:
# Définition du modèle LSTM pour le POS tagging
class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size): 
        super(LSTMTagger, self).__init__() # 
        self.hidden_dim = hidden_dim

        # 1. La couche d'Embedding : convertit les index des mots en vecteurs denses
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # 2. La couche LSTM : prend les embeddings en entrée et renvoie les états cachés
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # 3. La couche Linéaire : projette l'espace des états cachés vers l'espace des tags
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # Passage avant (forward pass) :
        # On transforme la phrase d'index en embeddings
        embeds = self.word_embeddings(sentence)
        
        # Le LSTM attend un tenseur à 3 dimensions : (longueur_sequence, taille_batch, taille_embedding)
        # On utilise view() pour ajouter la dimension de batch (qui est de 1 ici)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        
        # On passe les sorties du LSTM à la couche linéaire pour obtenir les scores de chaque tag
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        
        # On applique un LogSoftmax pour obtenir des log-probabilités
        tag_scores = torch.log_softmax(tag_space, dim=1)
        return tag_scores
    
"""
en gros , ce modèle LSTMTagger prend en entrée une phrase représentée par des indices de mots, la passe à travers une couche d'embedding, puis à travers un LSTM, 
et enfin projette les états cachés du LSTM vers l'espace des tags pour obtenir les scores de chaque tag.

ensuite , on applique une fonction LogSoftmax pour obtenir des log-probabilités pour chaque tag, ce qui est utile pour la classification multi-classes.

"""

"\nen gros , ce modèle LSTMTagger prend en entrée une phrase représentée par des indices de mots, la passe à travers une couche d'embedding, puis à travers un LSTM, \net enfin projette les états cachés du LSTM vers l'espace des tags pour obtenir les scores de chaque tag.\n\nensuite , on applique une fonction LogSoftmax pour obtenir des log-probabilités pour chaque tag, ce qui est utile pour la classification multi-classes.\n\n"

In [22]:
# 1. Définition des hyperparamètres (les dimensions de notre réseau)
EMBEDDING_DIM = 6  # Chaque mot sera représenté par un vecteur de taille 6
HIDDEN_DIM = 6     # Taille de la mémoire interne (état caché) du LSTM

# 2. Initialisation du modèle, de la fonction de perte (Loss) et de l'optimiseur
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss (idéal avec LogSoftmax)
optimizer = optim.SGD(model.parameters(), lr=0.1) # Descente de gradient stochastique

# 3. Test de prédiction AVANT entraînement
with torch.no_grad(): # On désactive le calcul des gradients (on ne fait pas d'entraînement ici)
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    
    print("Phrase testée :", training_data[0][0])
    print("\nScores bruts pour chaque tag (log-probabilités) :")
    print(tag_scores)
    
    # Pour chaque mot, on récupère l'index du tag qui a la plus grande probabilité
    _, predicted_tags = torch.max(tag_scores, dim=1)
    print("\nTags prédits (aléatoires pour l'instant) :", predicted_tags.tolist())
    
"""
ici , on teste le modèle avant l'entraînement pour voir les scores de log-probabilités pour chaque tag.
On utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.
On prépare la séquence d'entrée, on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, et enfin, on récupère les indices des tags prédits en prenant l'argmax de ces scores    
"""

Phrase testée : ['Amadou', 'étudie', 'le', 'bambara']

Scores bruts pour chaque tag (log-probabilités) :
tensor([[-1.2038, -0.8141, -1.3591],
        [-1.1921, -0.8750, -1.2745],
        [-1.2968, -0.8012, -1.2808],
        [-1.2716, -0.7597, -1.3791]])

Tags prédits (aléatoires pour l'instant) : [1, 1, 1, 1]


"\nici , on teste le modèle avant l'entraînement pour voir les scores de log-probabilités pour chaque tag.\nOn utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.\nOn prépare la séquence d'entrée, on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, et enfin, on récupère les indices des tags prédits en prenant l'argmax de ces scores    \n"

In [23]:
# On va entraîner le modèle sur 300 époques (passages complets sur le dataset)
print("Début de l'entraînement...")

for epoch in range(300):
    for sentence, tags in training_data:
        # Étape 1 : PyTorch accumule les gradients, on doit les remettre à zéro 
        # avant chaque phrase pour ne pas mélanger les calculs
        model.zero_grad()

        # Étape 2 : On prépare nos entrées (mots -> index) et nos cibles (tags -> index)
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        # Étape 3 : Passage avant (Forward pass) pour obtenir les prédictions
        tag_scores = model(sentence_in)

        # Étape 4 : Calcul de la perte (l'erreur commise par le modèle)
        loss = loss_function(tag_scores, targets)

        # Étape 5 : Rétropropagation (Backward pass) pour calculer les gradients
        loss.backward()

        # Étape 6 : Mise à jour des poids du modèle
        optimizer.step()
        
    # On affiche l'évolution de la perte toutes les 50 époques
    if (epoch + 1) % 50 == 0:
        print(f"Époque {epoch + 1}/300 | Perte (Loss) : {loss.item():.4f}")

print("Entraînement terminé !")

"""
en gros , ici , on entraîne le modèle sur 300 époques. Pour chaque phrase dans les données d'entraînement, on effectue les étapes suivantes :
1. On remet à zéro les gradients accumulés.
2. On prépare les entrées et les cibles en convertissant les mots et les tags en indices.
3. On passe la phrase à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.
4. On calcule la perte entre les prédictions du modèle et les tags réels.
5. On effectue la rétropropagation pour calculer les gradients.
6. On met à jour les poids du modèle en utilisant l'optimiseur.
On affiche la perte toutes les 50 époques pour suivre l'évolution de l'entraînement
    
"""

Début de l'entraînement...
Époque 50/300 | Perte (Loss) : 1.0094
Époque 100/300 | Perte (Loss) : 0.7223
Époque 150/300 | Perte (Loss) : 0.3642
Époque 200/300 | Perte (Loss) : 0.1315
Époque 250/300 | Perte (Loss) : 0.0650
Époque 300/300 | Perte (Loss) : 0.0403
Entraînement terminé !


"\nen gros , ici , on entraîne le modèle sur 300 époques. Pour chaque phrase dans les données d'entraînement, on effectue les étapes suivantes :\n1. On remet à zéro les gradients accumulés.\n2. On prépare les entrées et les cibles en convertissant les mots et les tags en indices.\n3. On passe la phrase à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.\n4. On calcule la perte entre les prédictions du modèle et les tags réels.\n5. On effectue la rétropropagation pour calculer les gradients.\n6. On met à jour les poids du modèle en utilisant l'optimiseur.\nOn affiche la perte toutes les 50 époques pour suivre l'évolution de l'entraînement\n\n"

In [24]:
# On repasse le modèle en mode évaluation
model.eval()

# 1. Test sur une phrase d'entraînement
phrase_connue = "Amadou étudie le bambara".split()
with torch.no_grad():
    inputs = prepare_sequence(phrase_connue, word_to_ix)
    scores = model(inputs)
    _, predictions = torch.max(scores, dim=1)
    
    # On reconstruit les noms des tags pour que ce soit lisible
    # On inverse le dictionnaire tag_to_ix pour avoir {index: "NOM"}
    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    tags_predits = [ix_to_tag[idx.item()] for idx in predictions]
    
    print("--- TEST 1 (Phrase connue) ---")
    print("Phrase :", phrase_connue)
    print("Tags prédits :", tags_predits)

print("\n" + "="*40 + "\n")

# 2. Test avec un mot totalement inconnu ("ordinateur")
# Notre dictionnaire ne connaît pas "ordinateur", il va donc le remplacer par <UNK>
phrase_inconnue = "Amadou comprend l' ordinateur".split() # (On sépare l'apostrophe pour simplifier)
with torch.no_grad():
    inputs_inc = prepare_sequence(phrase_inconnue, word_to_ix)
    scores_inc = model(inputs_inc)
    _, predictions_inc = torch.max(scores_inc, dim=1)
    tags_predits_inc = [ix_to_tag[idx.item()] for idx in predictions_inc]
    
    print("--- TEST 2 (Avec mot inconnu) ---")
    print("Phrase :", phrase_inconnue)
    print("Tags prédits :", tags_predits_inc)

--- TEST 1 (Phrase connue) ---
Phrase : ['Amadou', 'étudie', 'le', 'bambara']
Tags prédits : ['NOM', 'VERBE', 'DET', 'NOM']


--- TEST 2 (Avec mot inconnu) ---
Phrase : ['Amadou', 'comprend', "l'", 'ordinateur']
Tags prédits : ['NOM', 'VERBE', 'DET', 'DET']


# PADDING ET BATCH

In [25]:
# 1. On ajoute de nouvelles phrases de longueurs très différentes
training_data_multisize = [
    ("Amadou étudie le bambara".split(), ["NOM", "VERBE", "DET", "NOM"]),
    ("le modèle comprend les phrases".split(), ["DET", "NOM", "VERBE", "DET", "NOM"]), # Phrase plus longue
    ("il apprend".split(), ["PRON", "VERBE"]) # Phrase très courte
]

# 2. On reconstruit notre dictionnaire de mots
word_to_ix_pad = {}
# IMPORTANT : On réserve l'index 0 pour le token de Padding <PAD>
word_to_ix_pad["<PAD>"] = 0
word_to_ix_pad["<UNK>"] = 1

for sentence, tags in training_data_multisize:
    for word in sentence:
        if word not in word_to_ix_pad:
            word_to_ix_pad[word] = len(word_to_ix_pad)

# 3. Dictionnaire de tags (on réserve aussi l'index 0 pour le padding des tags si besoin)
tag_to_ix_pad = {"<PAD>": 0, "NOM": 1, "VERBE": 2, "DET": 3, "PRON": 4}

print("Nouveau dictionnaire de mots (avec <PAD> à l'index 0) :")
print(word_to_ix_pad)

"""
en résumé , ici , on a ajouté de nouvelles phrases de longueurs différentes pour enrichir notre jeu de données d'entraînement.
On a reconstruit le dictionnaire de mots pour inclure un token spécial de padding "<PAD>" à l'index 0, ce qui est utile pour gérer les séquences de longueurs variables lors de l'entraînement des modèles de réseaux de neurones.
On a également ajouté un token spécial "<UNK>" pour les mots inconnus, et on a mis à jour le dictionnaire des tags pour inclure un index de padding si nécessaire. Cela permet de mieux gérer les séquences de différentes longueurs et d'améliorer la robustesse du modèle lors de l'entraînement et de l'inférence.

"""

Nouveau dictionnaire de mots (avec <PAD> à l'index 0) :
{'<PAD>': 0, '<UNK>': 1, 'Amadou': 2, 'étudie': 3, 'le': 4, 'bambara': 5, 'modèle': 6, 'comprend': 7, 'les': 8, 'phrases': 9, 'il': 10, 'apprend': 11}


'\nen résumé , ici , on a ajouté de nouvelles phrases de longueurs différentes pour enrichir notre jeu de données d\'entraînement.\nOn a reconstruit le dictionnaire de mots pour inclure un token spécial de padding "<PAD>" à l\'index 0, ce qui est utile pour gérer les séquences de longueurs variables lors de l\'entraînement des modèles de réseaux de neurones.\nOn a également ajouté un token spécial "<UNK>" pour les mots inconnus, et on a mis à jour le dictionnaire des tags pour inclure un index de padding si nécessaire. Cela permet de mieux gérer les séquences de différentes longueurs et d\'améliorer la robustesse du modèle lors de l\'entraînement et de l\'inférence.\n\n'

In [ ]:
from torch.nn.utils.rnn import pad_sequence # pour le padding des séquences

# 1. Convertir toutes nos phrases et nos tags en listes de tenseurs d'index
sequences_mots = [] # pour stocker les séquences de mots converties en tenseurs
sequences_tags = [] # poir stocker les séquences de tags converties en tenseurs

for sentence, tags in training_data_multisize:
    # On utilise notre fonction de préparation avec le nouveau dictionnaire
    seq_tensor = prepare_sequence(sentence, word_to_ix_pad)
    tag_tensor = prepare_sequence(tags, tag_to_ix_pad)
    
    sequences_mots.append(seq_tensor)
    sequences_tags.append(tag_tensor)

# 2. Appliquer le Padding
# batch_first=True permet d'avoir une forme (taille_batch, longueur_sequence)
# padding_value=0 car 0 est l'index de <PAD>
padded_mots = pad_sequence(sequences_mots, batch_first=True, padding_value=0)
padded_tags = pad_sequence(sequences_tags, batch_first=True, padding_value=0)

# 3. Affichage du résultat
print("--- PHRASES RAMBOURRÉES (PADDED MOTS) ---")
print(padded_mots)
print("\nForme (Shape) du tenseur des mots :", padded_mots.shape)

print("\n" + "="*40 + "\n")

print("--- TAGS RAMBOURRÉS (PADDED TAGS) ---")
print(padded_tags)
print("\nForme (Shape) du tenseur des tags :", padded_tags.shape)

# les matrices viennent de la partie précédente, où nous avons converti les phrases et les tags en tenseurs d'index et appliqué un padding pour uniformiser la longueur des séquences.

"""
ici , on a converti toutes les phrases et leurs tags correspondants en tenseurs d'index, puis on a appliqué un padding pour que toutes les séquences aient la même longueur.
Le padding est fait avec l'index 0, qui correspond au token spécial "<PAD>".
Cela permet de créer des lots (batches) de données d'entrée et de sortie de taille uniforme, ce qui est nécessaire pour l'entraînement des modèles de réseaux de neurones.
"""

--- PHRASES RAMBOURRÉES (PADDED MOTS) ---
tensor([[ 2,  3,  4,  5,  0],
        [ 4,  6,  7,  8,  9],
        [10, 11,  0,  0,  0]])

Forme (Shape) du tenseur des mots : torch.Size([3, 5])


--- TAGS RAMBOURRÉS (PADDED TAGS) ---
tensor([[1, 2, 3, 1, 0],
        [3, 1, 2, 3, 1],
        [4, 2, 0, 0, 0]])

Forme (Shape) du tenseur des tags : torch.Size([3, 5])


In [ ]:
# 1. On redéfinit la fonction de perte pour ignorer le padding (index 0)
loss_function_pad = nn.NLLLoss(ignore_index=0)

# 2. Petite modification du modèle pour accepter les dimensions de batch
class LSTMTaggerWithBatch(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTaggerWithBatch, self).__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # padding_idx=0 met le vecteur du PAD à zéro
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True) # batch_first=True pour accepter la forme (batch, seq_len)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentences):
        # sentences a maintenant la forme: (batch_size, sequence_length)
        embeds = self.word_embeddings(sentences)
        
        # lstm_out aura la forme: (batch_size, sequence_length, hidden_dim)
        lstm_out, _ = self.lstm(embeds)
        
        # On passe toutes les sorties dans la couche linéaire
        tag_space = self.hidden2tag(lstm_out)
        
        # On applique le softmax sur la dernière dimension (les scores des tags)
        tag_scores = torch.log_softmax(tag_space, dim=-1)
        return tag_scores
    

"""
en gros , ce modèle LSTMTaggerWithBatch est une version modifiée du modèle précédent pour gérer les entrées par lots (batches).
La couche d'embedding est configurée pour ignorer le token de padding (index 0), et le LSTM est défini avec batch_first=True pour accepter des entrées de forme (batch_size, sequence_length, embedding_dim).
Le reste du modèle fonctionne de manière similaire, en projetant les sorties du LSTM vers l'espace des tags et en appliquant un LogSoftmax pour obtenir des log-probabilités pour chaque tag.

"""

In [28]:
# 1. Hyperparamètres
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

# 2. Initialisation du modèle avec batch et de l'optimiseur
model_batch = LSTMTaggerWithBatch(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=len(word_to_ix_pad),
    tagset_size=len(tag_to_ix_pad)
)

optimizer_batch = optim.SGD(model_batch.parameters(), lr=0.1)

# 3. Passage de notre batch de mots dans le modèle
with torch.no_grad():
    predictions_batch = model_batch(padded_mots)

print("Forme de notre batch d'entrée :", padded_mots.shape)
print("Forme des scores de sortie    :", predictions_batch.shape)

"""
en gros , ce code montre comment passer un batch de phrases (avec padding) à travers le modèle LSTMTaggerWithBatch.
On utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.
Le tenseur padded_mots contient les phrases avec padding, et predictions_batch contient les scores de log-probabilités pour chaque tag pour chaque mot dans le batch.
On affiche les formes (shapes) de ces tenseurs pour vérifier que tout est correct
La sortie doit avoir une forme de torch.Size([3, 5, 5]) (soit : [3 phrases, 5 mots par phrase, 5 étiquettes de tags possibles dans notre dictionnaire]).
    """

Forme de notre batch d'entrée : torch.Size([3, 5])
Forme des scores de sortie    : torch.Size([3, 5, 5])


"\nen gros , ce code montre comment passer un batch de phrases (avec padding) à travers le modèle LSTMTaggerWithBatch.\nOn utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.\nLe tenseur padded_mots contient les phrases avec padding, et predictions_batch contient les scores de log-probabilités pour chaque tag pour chaque mot dans le batch.\nOn affiche les formes (shapes) de ces tenseurs pour vérifier que tout est correct\nLa sortie doit avoir une forme de torch.Size([3, 5, 5]) (soit : [3 phrases, 5 mots par phrase, 5 étiquettes de tags possibles dans notre dictionnaire]).\n    "